In [ ]:
import geopandas as gpd
from IPython.display import display
import pandas as pd
import os
import numpy as np

In [ ]:
LIST_FILES = "../data/"

COLUMNS_TO_KEEP_ESPACE_VERT = ['nom_ev', 'categorie', 'adresse_libellevoie', 'adresse_codepostal', 'surface_totale_reelle', 'ouvert_ferme', 'geom']
COLUMNS_TO_KEEP_SIRET = ['siret', 'x', 'y', 'qualite_xy', 'plg_iris', 'plg_code_commune']
COLUMNS_TO_KEEP_IRIS = ['code_iris', 'nom_iris', 'typ_iris', 'nom_com', 'geometry']
COLUMNS_TO_KEEP_IRIS2 = ['typ_iris', 'code_iris', 'nom_iris', 'nom_com', 'insee_com', 'geometry']
COLUMNS_TO_KEEP_ARBRE = ['domanialite', 'arrondissement', 'libellefrancais', 'genre', 'espece', 'circonferenceencm', 'hauteurenm', 'stadedeveloppement', 'remarquable', 'geo_point_2d']
COLUMNS_TO_KEEP_LIEUX_MUNICIPAUX = ['id', 'name', 'categorie', 'address_street', 'address_postcode', 'address_city', 'accessibility', 'latitude', 'longitude', 'geo']
COLUMNS_TO_KEEP_LOYER = ['annee', 'id_zone', 'id_quartier', 'nom_quartier', 'piece', 'epoque', 'meuble_txt', 'max', 'min', 'ville', 'geometry']
COLUMNS_TO_KEEP_STATIONNEMENT_RESIDENCE = ['nom', 'st_area_shape', 'st_perimeter_shape', 'geo_point_2d', 'geometry']
COLUMNS_TO_KEEP_STATIONNEMENT_EMPLACEMENT = ['typsta', 'arrond', 'zoneres', 'locsta', 'numvoie', 'typevoie', 'nomvoie', 'parite', 'lon', 'lar', 'surface_calculee', 'geo_shape', 'geo_point_2d']
COLUMNS_TO_KEEP_VELO = ['stationcode', 'name', 'capacity', 'station_opening_hours', 'geometry']

COLUMNS_T0_KEEP = [COLUMNS_TO_KEEP_ESPACE_VERT + COLUMNS_TO_KEEP_SIRET + COLUMNS_TO_KEEP_IRIS + COLUMNS_TO_KEEP_IRIS2 + COLUMNS_TO_KEEP_ARBRE + COLUMNS_TO_KEEP_LIEUX_MUNICIPAUX + COLUMNS_TO_KEEP_LOYER + COLUMNS_TO_KEEP_STATIONNEMENT_RESIDENCE + COLUMNS_TO_KEEP_STATIONNEMENT_EMPLACEMENT + COLUMNS_TO_KEEP_VELO]


FILTRE_PARIS = ['75', 'paris']
COLUMNS_PARIS = ['dep', 'ville', 'address_city', 'arrondissement', 'plg_code_commune']


In [ ]:
file_load =[]
for files in os.listdir(LIST_FILES):
    if files.endswith(".parquet"):
        file_load.append(pd.read_parquet(os.path.join(LIST_FILES, files)))

    if files.endswith(".geojson"):
        file_load.append(gpd.read_file(os.path.join(LIST_FILES, files)))
    

In [ ]:
for i in file_load:
    display(i.dtypes)

In [ ]:
list_df = []

for df in file_load:
    for cols in COLUMNS_PARIS:
        if cols in df.columns and df[cols].astype(str).str.startswith(FILTRE_PARIS[0]).any():
            df = df[df[cols].astype(str).str.startswith(FILTRE_PARIS[0])]
        if cols in df.columns and df[cols].astype(str).str.lower().str.contains(FILTRE_PARIS[1], na=False).any():
            df = df[df[cols].astype(str).str.lower().str.contains(FILTRE_PARIS[1], na=False)]

    valid_cols = [c for c in COLUMNS_T0_KEEP[0] if c in df.columns]
    if len(valid_cols) > 0:
        list_df.append(df[valid_cols])
        display(df[valid_cols].head())

In [ ]:
def profile_dataframe(df):
    # Sécurité : Supprime les colonnes en double pour éviter les erreurs de type DataFrame vs Series
    df = df.loc[:, ~df.columns.duplicated()]
    
    profile = {}
    profile['num_rows'] = len(df)
    profile['num_columns'] = len(df.columns)
    profile['columns'] = {}
    
    for col in df.columns:
        col_profile = {}
        col_profile['dtype'] = str(df[col].dtype)
        col_profile['num_missing'] = df[col].isnull().sum()
        col_profile['num_unique'] = df[col].nunique()
        col_profile['min'] = df[col].min() if pd.api.types.is_numeric_dtype(df[col]) else None
        col_profile['max'] = df[col].max() if pd.api.types.is_numeric_dtype(df[col]) else None
        col_profile['mean'] = df[col].mean() if pd.api.types.is_numeric_dtype(df[col]) else None
        col_profile['std'] = df[col].std() if pd.api.types.is_numeric_dtype(df[col]) else None
        
        profile['columns'][col] = col_profile
    
    return profile

def global_profile(lst):
    return [profile_dataframe(df) for df in lst]

display([df for df in global_profile(list_df)])

BROUILLON 

In [ ]:
import numpy as np

L93 = 2154
SIRENE = "geoloc-geolocalisationetablissement-sirene-pour-etudes-statistiques-parquet.parquet"
ANNUAIRE = "fr-en-annuaire-education.geojson"
AIR_BRUIT = "Couches SIG air-bruit 2024_9_classes/AirBruit_2024.shp"
ARRONDISSEMENTS = [str(75101 + i) for i in range(20)]
FILES_TO_LOAD = ["espaces_verts.parquet", "iris.geojson", "les-arbres.parquet", "lieux-municipaux.parquet",
                 "logement-encadrement-des-loyers.geojson", "stationnement-sur-voie-publique-secteurs-residentiels.geojson",
                 "stationnement-voie-publique-emplacements.parquet", "velib-localisation-et-caracteristique-des-stations.geojson",
                 "fr-en-indicateurs-valeur-ajoutee-colleges.csv", "base-ic-logement-2022_csv/base-ic-logement-2022.CSV"]

LIEUX_FAMILLES = {  # 78 catégories → familles utiles pour un futur habitant
    "creche": r"Crèches|Haltes garderies|Jardins d'enfants",
    "sport": r"Gymnases|Piscines|Terrains|Tennis|Stades|Centres sportifs|Pistes|fitness|Skate|Patinoires|Bassins|Baignades",
    "culture": r"Bibliothèques|Musées|Théâtres|Conservatoires|Ateliers Beaux|Paris Anim|Pratiques Artistiques",
    "sante": r"[Ss]anté|PMI|vaccination|médico",
    "marche": r"Marchés",
    "mairie": r"Mairies|cartes d'identité|Centres administratifs",
}
EV_PARCS = ["Promenades ouvertes", "Bois"]  # espaces verts où l'on peut se promener
NIVEAUX = {"Ecole": "ecole", "Collège": "college", "Lycée": "lycee"}  # annuaire : public + privé, plus complet que les lieux municipaux
COLUMNS_VA_COLLEGES = {"UAI": "identifiant_de_l_etablissement", "Taux de réussite G": "college_taux_reussite", "VA du taux de réussite G": "college_va_reussite",
                       "Note à l'écrit G": "college_note_ecrit", "Taux d'accès 6eme 3eme": "college_taux_acces_6e_3e"}

LOGEMENT_PARTS = {  # part en % = somme(numérateurs) / somme(dénominateurs), base INSEE logement 2022
    "part_proprietaires_%": (["P22_RP_PROP"], ["P22_RP"]),
    "part_locataires_%": (["P22_RP_LOC"], ["P22_RP"]),
    "part_hlm_%": (["P22_RP_LOCHLMV"], ["P22_RP"]),
    "part_logements_vacants_%": (["P22_LOGVAC"], ["P22_LOG"]),
    "part_residences_secondaires_%": (["P22_RSECOCC"], ["P22_LOG"]),
    "part_1_2_pieces_%": (["P22_RP_1P", "P22_RP_2P"], ["P22_RP"]),
    "part_5_pieces_plus_%": (["P22_RP_5PP"], ["P22_RP"]),
    "part_100m2_plus_%": (["P22_RP_100120M2", "P22_RP_120M2P"], ["P22_RP"]),
    "part_avant_1946_%": (["P22_RP_ACH1919", "P22_RP_ACH1945"], ["P22_RP_ACHTOT"]),
    "part_2006_2018_%": (["P22_RP_ACH2019"], ["P22_RP_ACHTOT"]),
    "part_emmenages_10ans_plus_%": (["P22_MEN_ANEM10P"], ["P22_MEN"]),
    "part_menages_voiture_%": (["P22_RP_VOIT1P"], ["P22_RP"]),
    "part_menages_parking_%": (["P22_RP_GARL"], ["P22_RP"]),
    "part_suroccupation_%": (["C22_RP_SUROCC_MOD", "C22_RP_SUROCC_ACC"], ["C22_RP_NORME", "C22_RP_SUROCC_MOD", "C22_RP_SUROCC_ACC",
                                                                         "C22_RP_SOUSOCC_MOD", "C22_RP_SOUSOCC_ACC", "C22_RP_SOUSOCC_TACC"]),
}

def read(file, **kwargs):
    path = os.path.join(LIST_FILES, file)
    if file.endswith((".geojson", ".shp")):
        return gpd.read_file(path, **kwargs)
    if file.lower().endswith(".csv"):
        return pd.read_csv(path, sep=";", dtype={"IRIS": str, "COM": str, "UAI": str, "Code département": str}, **kwargs)
    return pd.read_parquet(path, **kwargs)

def to_geo(df, col=None, x=None, y=None, crs=4326):
    # DataFrame → GeoDataFrame Lambert 93, depuis une colonne WKB ou depuis x/y
    geometry = gpd.GeoSeries.from_wkb(df[col]) if col else gpd.points_from_xy(df[x], df[y])
    return gpd.GeoDataFrame(df.drop(columns=[col] if col else []), geometry=geometry, crs=crs).to_crs(L93)

In [ ]:
raw = {os.path.basename(file).rsplit(".", 1)[0]: read(file) for file in FILES_TO_LOAD}
raw["sirene"] = read(SIRENE, filters=[("plg_code_commune", "in", ARRONDISSEMENTS)])
raw["annuaire-education"] = read(ANNUAIRE, where="code_departement = '075'")
raw["air-bruit"] = read(AIR_BRUIT, bbox=tuple(raw["iris"].query("dep == '75'").to_crs(L93).total_bounds))

for name, df in raw.items():
    print(f"\n### {name} {df.shape}")
    display(pd.DataFrame(profile_dataframe(df)["columns"]).T)

In [ ]:
ev, arbres, lieux, emp, sirene = raw["espaces_verts"], raw["les-arbres"], raw["lieux-municipaux"], raw["stationnement-voie-publique-emplacements"], raw["sirene"]
annuaire, va, logement, air_bruit = raw["annuaire-education"], raw["fr-en-indicateurs-valeur-ajoutee-colleges"], raw["base-ic-logement-2022"], raw["air-bruit"]

checks = {
    "espaces verts : ouvert_ferme (7 graphies)": ev.ouvert_ferme.value_counts(dropna=False),
    "espaces verts : codes postaux hors Paris": ev.adresse_codepostal[~ev.adresse_codepostal.str.startswith("75", na=False)].value_counts(),
    "lieux : casse de address_city": lieux.address_city.value_counts().head(4),
    "lieux : ids présents dans 2 catégories (écoles polyvalentes, OK)": lieux.id.duplicated().sum(),
    "arbres : idbase dupliqués": arbres.idbase.duplicated().sum(),
    "arbres : hauteur = 0 / circonférence = 0": arbres[["hauteurenm", "circonferenceencm"]].eq(0).sum(),
    "arbres : hauteur > 40 m (aberrant)": arbres.hauteurenm.gt(40).sum(),
    "arbres : hors Paris (communes voisines)": arbres.arrondissement[~arbres.arrondissement.str.contains("PARIS|BOIS")].value_counts(),
    "loyers : lignes par (année, quartier, pièces, époque, meublé)": raw["logement-encadrement-des-loyers"].groupby(["annee", "id_quartier", "piece", "epoque", "meuble_txt"]).size().value_counts(),
    "sirene : plg_iris sur 4 chiffres seulement": sirene.plg_iris.dropna().str.len().value_counts(),
    "sirene : qualité du rattachement IRIS": sirene.qualite_iris.value_counts(),
    "annuaire : établissements sans coordonnées": annuaire.geometry.isna().sum(),
    "annuaire : types d'établissements": annuaire.type_etablissement.value_counts(),
    "collèges VA : sessions disponibles": va.Session.value_counts().sort_index(),
    "logement INSEE : IRIS de Paris retrouvés": logement.IRIS.isin(raw["iris"].code_iris[raw["iris"].dep == "75"]).sum(),
    "air-bruit : classes (bruit × air) dans l'emprise de Paris": air_bruit["9"].value_counts(),
    "colonnes 100% nulles": {name: df.columns[df.isna().all()].tolist() for name, df in raw.items() if df.isna().all().any()},
}
for check, result in checks.items():
    print(f"- {check} : {result.to_dict() if isinstance(result, pd.Series) else result}")

display(ev.groupby("type_ev").agg(n=("nom_ev", "size"), surface_mediane=("surface_totale_reelle", "median")))
display(emp.groupby("regpri").agg(emplacements=("plarel", "size"), places=("plarel", "sum")).sort_values("places", ascending=False))
display(arbres.groupby("domanialite").agg(n=("idbase", "size"), hauteur_moy=("hauteurenm", "mean")).sort_values("n", ascending=False))

In [ ]:
iris = raw["iris"].query("dep == '75'")[COLUMNS_TO_KEEP_IRIS].to_crs(L93).reset_index(drop=True)
iris["surface_km2"] = iris.area / 1e6

ev = to_geo(ev, "geom").drop(columns=["geom_x_y", "last_edited_user", "last_edited_date"])
ev["ouvert_24h"] = ev.ouvert_ferme.str.upper().str[0].map({"O": True, "N": False, "F": False})
ev["parc"] = ev.type_ev.isin(EV_PARCS)

arbres = to_geo(arbres.drop_duplicates("idbase"), "geo_point_2d")
arbres["hauteurenm"] = arbres.hauteurenm.where(arbres.hauteurenm.between(1, 40))
arbres["circonferenceencm"] = arbres.circonferenceencm.where(arbres.circonferenceencm > 0)
arbres["remarquable"] = arbres.remarquable.eq("OUI")

lieux = to_geo(lieux.dropna(subset=["geo"]), "geo")
lieux["address_city"] = lieux.address_city.str.title()
lieux["famille"] = np.select([lieux.categorie.str.contains(pattern) for pattern in LIEUX_FAMILLES.values()], list(LIEUX_FAMILLES), "autre")

ecoles = annuaire.dropna(subset=["geometry"]).to_crs(L93)
ecoles["niveau"] = ecoles.type_etablissement.map(NIVEAUX)
ecoles["prioritaire"] = ecoles.appartenance_education_prioritaire.notna()  # REP / REP+
va = va[(va.Session == va.Session.max()) & (va["Code département"].str.lstrip("0") == "75")]
ecoles = ecoles.dropna(subset=["niveau"]).merge(va[list(COLUMNS_VA_COLLEGES)].rename(columns=COLUMNS_VA_COLLEGES), how="left")

logement = logement[logement.IRIS.isin(iris.code_iris)].set_index("IRIS")

air_bruit = air_bruit.rename(columns={"9": "classe"}).to_crs(L93)
air_bruit["indice_bruit"] = air_bruit.classe // 10  # Symbologie.xlsx : dizaine = bruit (1 à 3), unité = air (1 à 3)
air_bruit["indice_air"] = air_bruit.classe % 10
air_bruit["point_noir"] = air_bruit.classe.eq(33)  # pollution et bruit au-dessus des valeurs limites

sirene = sirene.dropna(subset=["plg_iris"])
sirene["code_iris"] = sirene.plg_code_commune + sirene.plg_iris

clean = {
    "espaces_verts": ev,
    "arbres": arbres,
    "lieux": lieux,
    "velib": raw["velib-localisation-et-caracteristique-des-stations"].drop(columns="station_opening_hours").to_crs(L93),
    "stationnement": to_geo(emp, "geo_point_2d").drop(columns=["geo_shape", "numilot", "numiris", "prefet"]),
    "zones_residentielles": raw["stationnement-sur-voie-publique-secteurs-residentiels"][["nom", "geometry"]].rename(columns={"nom": "zone_residentielle"}).to_crs(L93),
    "loyers": raw["logement-encadrement-des-loyers"].astype({"annee": int}).to_crs(L93),
    "sirene": sirene[["siret", "code_iris", "qualite_iris", "x", "y"]],
    "ecoles": ecoles[[*COLUMNS_VA_COLLEGES.values(), "nom_etablissement", "niveau", "statut_public_prive", "prioritaire", "geometry"]],
    "logement": logement,
    "air_bruit": air_bruit,
}
print("sirene : code_iris reconstruit présent dans les IRIS :", clean["sirene"].code_iris.isin(iris.code_iris).mean())
print("lieux par famille :", clean["lieux"].famille.value_counts().to_dict())
for name, df in clean.items():
    print(name, df.shape)

# Passage au grain IRIS

In [ ]:
iris_shapes = iris[["code_iris", "geometry"]]
centres = iris.set_geometry(iris.representative_point())[["code_iris", "geometry"]]

def count_by_iris(points, name, by=None, val=None):
    # compte (ou somme `val`) des points par IRIS, éventuellement éclaté par `by`
    grouped = gpd.sjoin(points, iris_shapes, predicate="within").groupby(["code_iris"] + ([by] if by else []))
    result = (grouped[val].sum() if val else grouped.size()).rename(name)
    return result.unstack(fill_value=0).add_prefix(f"{name}_") if by else result

def nearest(shapes, points):
    return gpd.sjoin_nearest(shapes, points[["geometry"]], distance_col="d").groupby("code_iris").d.min()

def distance(points, name):
    # 0 si au moins un point dans l'IRIS, sinon distance centre de l'IRIS → point le plus proche
    return nearest(centres, points).where(nearest(iris_shapes, points) > 0, 0).round().rename(f"dist_{name}_m")

def from_shapes(shapes, cols, nearest=False):
    # valeur du polygone qui contient le centre de l'IRIS (ou, si nearest, du point le plus proche du centre)
    joined = gpd.sjoin_nearest(centres, shapes[cols + ["geometry"]]) if nearest else gpd.sjoin(centres, shapes[cols + ["geometry"]], predicate="within")
    return joined.drop_duplicates("code_iris").set_index("code_iris")[cols]

def weighted_mean(shapes, cols):
    # moyenne de `cols` dans chaque IRIS, pondérée par la surface de l'intersection
    pieces = gpd.overlay(iris_shapes, shapes[cols + ["geometry"]])
    return pieces[cols].mul(pieces.area, axis=0).groupby(pieces.code_iris).sum().div(pieces.area.groupby(pieces.code_iris).sum(), axis=0)

In [ ]:
lieux_utiles = lieux.query("famille != 'autre'")
colleges = ecoles.dropna(subset=["college_taux_reussite"])
parcs = ev.query("parc")

# loyers (grain quartier) : 2025 = dernière année, ref €/m² par nb de pièces (non meublé) + moyenne meublé et évolution 2019→2025
loyers = clean["loyers"]
loyers_2025 = loyers.query("annee == 2025")
loyers_by_year = loyers.pivot_table(index="id_quartier", columns="annee", values="ref")
loyers_q = gpd.GeoDataFrame(
    loyers_2025.query("meuble_txt == 'non meublé'").pivot_table(index="id_quartier", columns="piece", values="ref").add_prefix("loyer_ref_").add_suffix("p")
    .join([loyers_2025.pivot_table(index="id_quartier", columns="meuble_txt", values="ref").rename(columns={"meublé": "loyer_ref_meuble", "non meublé": "loyer_ref_non_meuble"}),
           (loyers_by_year[2025] / loyers_by_year[2019] - 1).mul(100).rename("loyer_evol_2019_2025_%"),
           loyers.drop_duplicates("id_quartier").set_index("id_quartier")[["nom_quartier", "geometry"]]]),
    crs=L93)

part_espaces_verts = iris.set_index("code_iris").intersection(ev.union_all()).area / iris.set_index("code_iris").area * 100

logement = clean["logement"]
logement_parts = pd.DataFrame({name: logement[num].sum(axis=1) / logement[den].sum(axis=1).replace(0, np.nan) * 100
                               for name, (num, den) in LOGEMENT_PARTS.items()}).round(1)

air_bruit_iris = weighted_mean(air_bruit, ["indice_air", "indice_bruit", "point_noir"]).round(2)
air_bruit_iris["point_noir"] *= 100

features = [
    count_by_iris(clean["velib"], "nb_velib_stations"),
    count_by_iris(clean["velib"], "velib_bornes", val="capacity"),
    count_by_iris(lieux_utiles, "nb", by="famille"),
    *[distance(lieux_utiles[lieux_utiles.famille == famille], famille) for famille in LIEUX_FAMILLES],
    count_by_iris(ecoles, "nb", by="niveau"),
    count_by_iris(ecoles.query("prioritaire"), "nb_etablissements_rep"),
    *[distance(ecoles[ecoles.niveau == niveau], niveau) for niveau in NIVEAUX.values()],
    from_shapes(colleges, list(COLUMNS_VA_COLLEGES.values())[1:], nearest=True),
    distance(clean["velib"], "velib"),
    distance(parcs, "parc"),
    count_by_iris(arbres, "nb_arbres"),
    gpd.sjoin(arbres, iris_shapes, predicate="within").groupby("code_iris").agg(arbres_hauteur_moy=("hauteurenm", "mean"), arbres_remarquables=("remarquable", "sum")),
    count_by_iris(clean["stationnement"], "places", by="regpri", val="plarel").rename(columns=lambda c: c.lower().replace(" ", "_").replace("/", "_")),
    clean["sirene"].groupby("code_iris").size().rename("nb_etablissements"),
    part_espaces_verts.rename("part_espaces_verts_%"),
    count_by_iris(parcs.set_geometry(parcs.representative_point()), "nb_parcs"),
    from_shapes(loyers_q, [c for c in loyers_q.columns if c.startswith("loyer")] + ["nom_quartier"]),
    from_shapes(clean["zones_residentielles"], ["zone_residentielle"]),
    logement["P22_LOG"].round().rename("nb_logements"),
    logement_parts,
    air_bruit_iris.rename(columns={"indice_air": "indice_air_moy", "indice_bruit": "indice_bruit_moy", "point_noir": "part_point_noir_air_bruit_%"}),
]

In [ ]:
dataset = iris.set_index("code_iris").join(features)
counts = [c for c in dataset.columns if c.startswith(("nb_", "velib_", "places_", "arbres_remarquables"))]
dataset[counts] = dataset[counts].fillna(0).astype(int)
dataset = dataset.join(dataset[["nb_etablissements", "nb_arbres", "velib_bornes"]].div(dataset.surface_km2, axis=0).add_suffix("_par_km2").round(1)).reset_index()

print(dataset.shape)
display(pd.DataFrame(profile_dataframe(dataset.drop(columns="geometry"))["columns"]).T)

In [ ]:
os.makedirs(LIST_FILES + "clean", exist_ok=True)
for name, df in clean.items():
    df.to_parquet(LIST_FILES + f"clean/{name}.parquet")
dataset.to_parquet(LIST_FILES + "iris_paris_dataset.parquet")  # GeoParquet (avec géométrie, pour les cartes)
dataset.drop(columns="geometry").to_csv(LIST_FILES + "iris_paris_dataset.csv", index=False)